In [1]:
"""
"""

# imports
import os, sys

# import the __functions.py (custom functions)
sys.path.append(os.getcwd()) # add code folder to system path
from __functions import *  # imports all custom functions

# local data directories
boxdir = '/Users/mcc/Library/CloudStorage/Box-Box/MCC/'
projdir = os.path.dirname(os.getcwd())
print(projdir)

proj = 'EPSG:5070'

print("Complete!")

/Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation
Complete!


In [2]:
# --- Load the latest incidents table joined to fire perimeters
fp = os.path.join(projdir, 'data/spatial/mod/ics209plus_2014to2023_perimeters.gpkg')
v2_all = gpd.read_file(fp)
v2_all = v2_all[~v2_all['FUEL_MODEL'].isna()]
v2_all = v2_all[v2_all['START_YEAR'] <= 2022] # temporal filter
# --- Filter to timber only fires between 2014-2022
v2_timber = v2_all[v2_all['FUEL_MODEL'].str.contains('timber', case=False)] # Timber fuel model filter
print(v2_timber['FUEL_MODEL'].value_counts())
print(f"There are {len(v2_timber)} incidents in the updated table (2014-2022), timber fuel model).")

FUEL_MODEL
Timber (Litter and Understory)    1306
Timber (Grass and Understory)     1260
Closed Timber Litter                98
Name: count, dtype: int64
There are 2664 incidents in the updated table (2014-2022), timber fuel model).


In [3]:
# --- Load the MTBS burn severity model dataframe (Herpe et al., Nov 2025)
fp = os.path.join(projdir, "data/tabular/incidents/xlsx/WFC data as of 11-6-2025--V4--trimmed.xlsx")
v1_sev = pd.read_excel(fp)
print(f"Severity model subset: {len(v1_sev)} incidents.")
print(f"Matching MTBS_IDs in V2: {len(v2_timber[v2_timber['MTBS_ID'].isin(v1_sev['Event_ID'].unique())])} incidents.\n")

# --- Add columns for acre difference
v1_sev['ACRE_DIFF'] = v1_sev['FINAL_ACRES'] - v1_sev['BurnBndAc']
v1_sev['ACRE_DIFF_abs'] = v1_sev['ACRE_DIFF'].abs()

# --- Mark incidents that are not in the updated table
m = v1_sev[~v1_sev['Event_ID'].isin(v2_timber['MTBS_ID'].unique())].copy()
v1_sev['CHECK'] = v1_sev['Event_ID'].isin(m['Event_ID'].unique())
print(len(v1_sev[v1_sev['CHECK'] == True]))

# --- Save this back out
out_fp = os.path.join(projdir, "data/tabular/incidents/xlsx/WFC data as of 11-6-2025--V4--trimmed_MC.xlsx")
v1_sev.to_excel(out_fp)
print(f"Overwriting file to {out_fp}.\n")

# check on the absolute acre difference
v1_sev['ACRE_DIFF_abs'].describe()

Severity model subset: 1092 incidents.
Matching MTBS_IDs in V2: 1073 incidents.

22
Overwriting file to /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/tabular/incidents/xlsx/WFC data as of 11-6-2025--V4--trimmed_MC.xlsx.



count      1092.000000
mean       2419.730018
std       22513.049399
min           0.000000
25%          78.000000
50%         244.500000
75%         781.250000
max      677309.000000
Name: ACRE_DIFF_abs, dtype: float64

In [4]:
# --- Load the extended dataset (to include all timber fires)
fp = os.path.join(projdir,"data/tabular/incidents/csv/ics209plus-wf_incidents_2014to2022-draft_111825.csv")
v1_add = pd.read_csv(fp)
print(f"There are {len(v1_add)} incidents in the table for additional fires.")

# --- Subset to incidents with a matching perimeter
v2_timber_add = v2_timber[v2_timber['INCIDENT_ID'].isin(v1_add['INCIDENT_ID'].unique())]
print(f"Matching INCIDENT_IDs in V2: {len(v2_timber_add)} incidents.\n")
print(f"Acres distribution:\n{v2_timber_add['FINAL_ACRES'].describe()}")

There are 4655 incidents in the table for additional fires.
Matching INCIDENT_IDs in V2: 2017 incidents.

Acres distribution:
count      2017.000000
mean       8719.733171
std       34563.952746
min           2.700000
25%         235.000000
50%        1100.000000
75%        4093.800000
max      963309.000000
Name: FINAL_ACRES, dtype: float64


In [5]:
# --- Check the difference in columns:
print(f"Columns in the v1_add: {len(v1_add.columns)}")
print(f"Columns in the v2_timber_all: {len(v2_timber.columns)}")
matching_cols = set(v1_add.columns) & set(v2_timber.columns)
print("Matching columns:", len(matching_cols))
# Columns in df1 but NOT in df2
only_in_df1 = set(v2_timber.columns) - set(v1_add.columns)
print("Only in df1:", only_in_df1)

Columns in the v1_add: 81
Columns in the v2_timber_all: 92
Matching columns: 80
Only in df1: {'STUSPS', 'na_l3code', 'FOD_C', 'geometry', 'na_l3name', 'IRWIN_C', 'date_diff_days', 'acre_diff_abs', 'match_key', 'match_method', 'acre_diff_pct', 'WUMI_MTBS_ID'}


In [6]:
# --- Check on incidents not in the v1_add
v2_timber_ = v2_timber[v2_timber['START_YEAR'] <= 2022].copy() # make sure we have 2014-2022
print(f"Number of timber incidents: {len(v2_timber)}")
id_counts = v2_timber_['INCIDENT_ID'].value_counts()
duplicates = id_counts[id_counts > 1]
print(f"Duplicate IDs: {len(duplicates)}")
print(v2_timber_['START_YEAR'].min(), v2_timber_['START_YEAR'].max())

Number of timber incidents: 2664
Duplicate IDs: 0
2014 2022


In [7]:
# --- Load the latest ICS209-PLUS spatial
fp = os.path.join(boxdir,'projects/ics209plus/data/spatial/mod/ics209plus-spatial_1999to2023.gpkg') # update, spatial
incis = gpd.read_file(fp)
incis = incis[(incis['START_YEAR'] >= 2014) & (incis['START_YEAR'] <= 2022)] # temporal filter
incis = incis[incis['FUEL_MODEL'].isin(v2_timber_['FUEL_MODEL'].unique())] # fuel model filter
incis.drop(columns=['geometry'], inplace=True) # drop geometry
print(f"There are [{len(incis)}] incidents in the table (CONUS, 2014-2022, timber fuel models).")

There are [5499] incidents in the table (CONUS, 2014-2022, timber fuel models).


In [8]:
print(len(incis[incis['INCIDENT_ID'].isin(v2_timber_['INCIDENT_ID'].unique())]))

2664


In [9]:
# --- Merge into the full dataframe
inci_ids = v2_timber_['INCIDENT_ID'].unique()
v2_timber_all = incis.copy() # work from a copy of all timber fires (2014-2022)
v2_timber_all['PERIMETER'] = v2_timber_all['INCIDENT_ID'].isin(inci_ids) # mark where we have perimeters
print(v2_timber_all[['INCIDENT_ID','PERIMETER']].head())
print(f"\nIncidents with perimeters: {len(v2_timber_all[v2_timber_all['PERIMETER'] == True])}")
print(f"Total incidents: {len(v2_timber_all)}\n")

# --- Join in the match_method
v2_timber_all = pd.merge(v2_timber_all, v2_timber[['INCIDENT_ID','match_method']], on='INCIDENT_ID', how='left')

# --- Handle columns
v1_add.drop(columns=['Unnamed: 0'], inplace=True)
cols_to_keep = list(v1_add.columns)+ ['PERIMETER','match_method']
v2_timber_all = v2_timber_all[cols_to_keep]
print(v2_timber_all.columns)

# --- Keep only fuel models that were in the severity subset
v2_timber_all = v2_timber_all[v2_timber_all['FUEL_MODEL'].isin(v1_sev['FUEL_MODEL'].unique())]
print(len(v2_timber_all))

# --- Save this file out as the add file
out_fp = os.path.join(projdir,'data/tabular/incidents/xlsx/ics209plus-wf_incidents_2014to2022-draft_timber_only_111825.xlsx')
v2_timber_all.to_excel(out_fp)
print(f"Saved to {out_fp}")

                     INCIDENT_ID  PERIMETER
23152        2014_1006699_MURPHY      False
23154    2014_1020132_CASPER TWO      False
23155  2014_1021954_YAXON CANYON      False
23157        2014_1029599_LADD 2      False
23161        2014_1033753_COBURN      False

Incidents with perimeters: 2664
Total incidents: 5499

Index(['CAUSE', 'COMPLEX', 'DISCOVERY_DATE', 'DISCOVERY_DOY',
       'EVACUATION_REPORTED', 'EXPECTED_CONTAINMENT_DATE', 'FATALITIES',
       'FATALITIES_PUBLIC', 'FATALITIES_RESPONDER', 'FINAL_ACRES',
       'FINAL_REPORT_DATE', 'FIRE_SIZE_CLASS', 'FIRE_YEAR', 'FOD_CONT_DATE',
       'FOD_CONT_DOY', 'FOD_CONT_TIME', 'FOD_DISCOVERY_DATE',
       'FOD_DISCOVERY_DOY', 'FOD_DISCOVERY_TIME', 'FOD_EXCL_CAT',
       'FOD_FIRE_SIZE', 'FOD_ID', 'FOD_INCIDENT_GROUP_NAME', 'FOD_LATITUDE',
       'FOD_LONGITUDE', 'FUEL_MODEL', 'INCIDENT_DESCRIPTION', 'INCIDENT_ID',
       'INCIDENT_ID_OLD', 'INCIDENT_NAME', 'INCIDENT_NUMBER',
       'INCTYP_ABBREVIATION', 'INC_IDENTIFIER', 'INC_MGMT

In [12]:
# --- Load the summary tables and subset appropriately
nsi = pd.read_csv(os.path.join(projdir,"data/tabular/summaries/NSI_summary.csv"))
wrc = pd.read_csv(os.path.join(projdir,'data/tabular/summaries/WRC_hurisk_popden_weighted.csv'))
twig = pd.read_csv(os.path.join(projdir, 'data/tabular/summaries/TWIG_summary_wide-format.csv'))

v2_timber_perims = v2_timber_all[v2_timber_all['PERIMETER'] == True]
print(len(v2_timber_all))
print(len(v2_timber_perims))

# --- Subset to the larger dataframe
nsi = nsi[nsi['INCIDENT_ID'].isin(v2_timber_['INCIDENT_ID'].unique())].reset_index(drop=True)
print(len(nsi))
wrc = wrc[wrc['INCIDENT_ID'].isin(v2_timber_['INCIDENT_ID'].unique())].reset_index(drop=True)
print(len(wrc))
twig = twig[twig['INCIDENT_ID'].isin(v2_timber_['INCIDENT_ID'].unique())].reset_index(drop=True)
print(len(twig))

# --- Save these out
nsi.to_excel(os.path.join(projdir,'data/tabular/summaries/NSI_summary_2014to2022.xlsx'))
wrc.to_excel(os.path.join(projdir,'data/tabular/summaries/TWIG_summary_2014to2022.xlsx'))
twig.to_excel(os.path.join(projdir,'data/tabular/summaries/WRC_summary_2014to2022.xlsx'))

5298
2566
852
Index(['INCIDENT_ID', 'val_struct_NR', 'val_struct_R', 'n_struct_NR',
       'n_struct_R', 'n_struct_total', 'prop_R', 'val_struct_NR24',
       'val_struct_R24'],
      dtype='object')
2664
998


In [ ]:
v2_timber_all['FUEL_MODEL'].value_counts()

In [ ]:
twig.columns

In [ ]:
twig[twig['Hand Pile'].isna()]

In [ ]:
# --- Make sure all the original severity model incidents are in the new data
incis_in_v1_sev = incis[incis['MTBS_ID'].isin(v1_sev['Event_ID'].unique())]
print(f"Matching incidents by MTBS ID: {len(incis_in_v1_sev)}")
print(f"Matching incidents by MTBS ID: {incis_in_v1_sev['MTBS_ID'].nunique()}")

In [ ]:
# --- Check on representation of the updated dataset in the OG
# for severity model
v1_mtbs_ids = v1_sev['Event_ID'].unique()
print(f"Matching MTBS_IDs (severity model): {len(v2_all[v2_all['MTBS_ID'].isin(v1_mtbs_ids)])} (of {len(v1_mtbs_ids)}).")

In [ ]:
# --- See how many version 2 joins are in the table that Evan sent
v2_in_v1_add = v1_add[v1_add['INCIDENT_ID'].isin(v2_timber['INCIDENT_ID'].unique())]
print(len(v2_in_v1_add))

In [ ]:
print(f"Matching MTBS_IDs (severity model): {len(v2_in_v1_add[v2_in_v1_add['MTBS_ID'].isin(v1_mtbs_ids)])} (of {len(v1_mtbs_ids)}).")

In [ ]:
v2_in_v1_add['START_YEAR'].max()

In [ ]:
# --- Check on the acre differences in the primary dataframe
m = v1_sev[~v1_sev['Event_ID'].isin(v2_all['MTBS_ID'].unique())].copy()
print(f"Number of potential mismatches: {len(m)}\n")

# calculate the acre difference
m['ACRE_DIFF'] = m['FINAL_ACRES'] - m['BurnBndAc']
m['ACRE_DIFF_abs'] = m['ACRE_DIFF'].abs()

# symmetric percent diff (mean denominator) to avoid oddities
den = (m['FINAL_ACRES'].abs() + m['BurnBndAc'].abs()) / 2.0
m['ACRE_DIFF_pct'] = 100.0 * m['ACRE_DIFF_abs'] / den.replace(0, np.nan)

# -- Make a flag for these
# check on the 70th percentile
qt = m['ACRE_DIFF_pct'].quantile(0.70)
print(f"90th percentile absolute acre difference (%): {qt}\n")
m[['Event_ID','Incid_Name','INCIDENT_NAME','BurnBndAc','FINAL_ACRES','ACRE_DIFF_abs','ACRE_DIFF_pct']].head(len(m))

In [ ]:
# for larger subset
v1_all_ids = v1_add['INCIDENT_ID'].unique()
print(f"Matching INCIDENT_IDs (treatment model): {len(v2_all[v2_all['INCIDENT_ID'].isin(v1_all_ids)])} (of {len(v1_all_ids)}).")

In [ ]:
# --- See how many 2023 fires there are
v2_23 = v2_all[v2_all['START_YEAR']>=2023]
print(f"{len(v2_23)} additional incidents (2023).")
print(f"\tWith MTBS_ID: {len(v2_23[v2_23['MTBS_ID'].isna()])}")

In [ ]:
# --- Check on the column mismatch

In [ ]:
v2_23.head()

In [ ]:
# --- Load the Herpe et al. version of the ICS209
fp = os.path.join(projdir, "data/tabular/incidents/ics209plus-wf_incidents_2014to2022-draft_111825.csv")
v1 = pd.read_csv(fp)
print(f"There are {len(v1)} incidents in the table.")
print(f"\n{v1.columns}")

# --- Load the updated (spatial points) through 2023
fp = os.path.join(boxdir,'projects/ics209plus/data/spatial/mod/ics209plus-spatial_1999to2023.gpkg')
v2 = gpd.read_file(fp)
v2 = v2[(v2['START_YEAR'] >= 2014) & (v2['START_YEAR'] <= 2022)] # temporal filter to align with version 1
print(f"\nThere are {len(v2)} incidents in the table.")
print(f"\n{v2.columns}")

In [ ]:
v1['FUEL_MODEL'].unique()

In [ ]:
len(v1)

In [ ]:
# --- Compare columns
cols_v1 = set(v1.columns)
cols_v2 = set(v2.columns)

only_in_v1 = sorted(cols_v1 - cols_v2)
only_in_v2 = sorted(cols_v2 - cols_v1)
common_cols = sorted(cols_v1 & cols_v2)

print("\nColumns only in v1:")
for c in only_in_v1:
    print(f"  - {c}")

print("\nColumns only in v2:")
for c in only_in_v2:
    print(f"  - {c}")

print(f"\nNumber of common columns: {len(common_cols)}")

In [ ]:
# --- Check FUEL_MODEL
print(v1['FUEL_MODEL'].unique())
v2_fm = v2[v2['FUEL_MODEL'].isin(v1['FUEL_MODEL'].unique())]
print(len(v2_fm))
print(v2_fm['FUEL_MODEL'].unique())

In [ ]:
v1.columns

In [ ]:
# --- Check on the structures threatened / destroyed and evacuations
print(f"Structures threatened (v1): {v1['STR_THREATENED_MAX'].sum()}")
print(f"Structures threatened (v2): {v2_fm['STR_THREATENED_MAX'].sum()}")
print(f"Structures destroyed (v1): {v1['STR_DESTROYED_TOTAL'].sum()}")
print(f"Structures destroyed (v2): {v2_fm['STR_DESTROYED_TOTAL'].sum()}")